# FLUX.2 Small Decoder (AutoencoderKLFlux2): Inference Examples

In [ ]:
import json
import boto3

### Query endpoint

In [ ]:
endpoint_name = "!!!name!!!"  # Replace with your endpoint name
region = "!!!region!!!"  # Replace with your endpoint's AWS region


def query_endpoint(payload):
    client = boto3.client("runtime.sagemaker", region_name=region)
    response = client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(payload).encode("utf-8"),
    )
    result = json.loads(response["Body"].read())
    print(json.dumps(result, indent=2, default=str))
    return result

### 1. Health check — decode a random latent

The FLUX.2 small decoder is a distilled VAE. With no input image, the endpoint samples a random latent tensor and runs the decoder to produce a full-resolution image. This is the fastest way to confirm the endpoint is live and the decoder path works.

In [ ]:
payload = {'parameters': {'height': 256, 'width': 256, 'seed': 0}}

In [ ]:
payload = {"parameters": {"height": 256, "width": 256, "seed": 0}}
resp = query_endpoint(payload)
print("mode:", resp["mode"])
# resp["image"] is a base64-encoded PNG
print("image bytes (base64 len):", len(resp["image"]))

### 2. Decode with an image-conditioned request

Send a base64-encoded image alongside the request. The endpoint samples/decodes a latent and returns a base64-encoded PNG in the response. The response reports its 'mode' and the resulting 'image'; only the keys the endpoint actually returns are read here.

In [ ]:
payload = {'image': 'iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAABC8jL9AAAAWklEQVR4nO3PMQ0AIAwEwEIQhBz8i0FKQpqbT9zY7t7HYnb1gLcMSGYCyUwgmQkkM4FkJpDMBJKZQDITSGYCyUwgmQkkM4FkJpDMBJKZQDITSGYCyUwg2QP0kQGRlp1BvwAAAABJRU5ErkJggg==',
 'parameters': {'seed': 0}}

In [ ]:
import base64
# Use an inline base64-encoded PNG (no local file needed)
img_b64 = "iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAABC8jL9AAAAWklEQVR4nO3PMQ0AIAwEwEIQhBz8i0FKQpqbT5zY7t7HYnb1gLcMSGYCyUwgmQkkM4FkJpDMBJKZQDITSGYCyUwgmQkkM4FkJpDMBJKZQDITSGYCyUwg2QP0kQGRlp1BvwAAAABJRU5ErkJggg=="
payload = {"image": img_b64, "parameters": {"seed": 0}}
resp = query_endpoint(payload)
print("mode:", resp["mode"])
# resp["image"] is a base64-encoded PNG; decode it to view the output
out_png = base64.b64decode(resp["image"])
print("decoded PNG bytes:", len(out_png))

### 3. Higher-resolution latent decode

Decode a larger random latent to a 512x512 image. Because the small decoder uses narrower channel widths ([96,192,384,384] vs [128,256,512,512]), it decodes larger latents with less VRAM, enabling higher resolutions.

In [ ]:
payload = {'parameters': {'height': 512, 'width': 512, 'seed': 42}}

In [ ]:
payload = {"parameters": {"height": 512, "width": 512, "seed": 42}}
resp = query_endpoint(payload)
print("mode:", resp["mode"])
print("image bytes (base64 len):", len(resp["image"]))

### 4. Deterministic reproducibility with seeds

The endpoint accepts a seed parameter. Reusing the same seed on the same input yields a deterministic latent sample and therefore a reproducible output — useful for regression testing and A/B comparisons against the full FLUX.2 decoder.

In [ ]:
payload = {'parameters': {'height': 256, 'width': 256, 'seed': 123}}

In [ ]:
a = query_endpoint({"parameters": {"height": 256, "width": 256, "seed": 123}})
b = query_endpoint({"parameters": {"height": 256, "width": 256, "seed": 123}})
print("mode:", a["mode"])
print("reproducible:", a["image"] == b["image"])